[Reference](https://medium.com/@pankaj_pandey/58e50e272e5c?sk=3a7bd049aa5c1c70a06f0d0844293969$0)

```
from langchain.agents import create_agent
# Model + tools, no controls. Demos fine, fails in production.
agent = create_agent("anthropic:claude-sonnet-4-5", tools=[search, send_email, delete_record])
```

In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    wrap_tool_call,
    HumanInTheLoopMiddleware,
    LLMToolSelectorMiddleware,
)

# Skill routing: let a fast model pick only the relevant tools per request,
# instead of exposing all of them every call.
tool_selector = LLMToolSelectorMiddleware(model="anthropic:claude-haiku-4-5")

# Verification gate: block a risky tool unless a condition is met.
@wrap_tool_call
def guard(request, handler):
    if request.tool_call["name"] == "delete_record":
        return "Blocked: delete requires an approved ticket."   # short-circuit
    return handler(request)

# Governance: pause for human approval before irreversible actions.
approval = HumanInTheLoopMiddleware(
    interrupt_on={"send_email": True, "delete_record": True}
)

agent = create_agent(
    "anthropic:claude-sonnet-4-5",
    tools=[search, send_email, delete_record],
    middleware=[tool_selector, guard, approval],
)

```
User or system trigger
        |
Intent router
        |
Policy and permission check
        |
Context builder  (RAG / search, memory, session state, tool metadata)
        |
Planner / agent loop
        |
Tool router  (internal APIs, DB/SQL, MCP tools, filesystem, sandbox)
        |
Verifier  (schema, citations, business rules, safety)
        |
Human approval gate  (only if risky)
        |
Final response or action
        |
Trace + eval + feedback dataset
```